# Milestone 3 - Modelação e Avaliação

Neste notebook é realizada a fase de modelação do dataset IBM HR Analytics Employee Attrition & Performance, com o objetivo de desenvolver, treinar e avaliar modelos de aprendizagem automática capazes de prever a probabilidade de abandono de colaboradores (Attrition).

Esta etapa enquadra-se nas fases de Modelling e Evaluation da metodologia CRISP-DM, dando continuidade ao trabalho desenvolvido na Milestone 2, onde os dados foram explorados, limpos e preparados para modelação.

Após a análise exploratória e engenharia de atributos realizadas anteriormente, esta fase tem como propósito:

- Definir a estratégia de modelação, incluindo a divisão dos dados em conjuntos de treino e teste;
- Selecionar métricas de avaliação adequadas ao problema de negócio, com especial foco na identificação da classe minoritária (Attrition = Yes);
- Implementar um modelo baseline para estabelecer um ponto de referência de desempenho;
- Treinar e comparar diferentes algoritmos de classificação supervisionada;
- Aplicar técnicas de validação cruzada (Cross-Validation) para garantir a robustez dos resultados;
- Otimizar os modelos através de ajuste de hiperparâmetros (Hyperparameter Tuning);
- Avaliar o desempenho dos modelos com base em métricas relevantes (Recall, F1-score, ROC-AUC) e na análise da matriz de confusão;
- Analisar a importância das variáveis (Feature Importance), identificando os principais fatores que influenciam a previsão de atrito;
- Diagnosticar o comportamento dos modelos, identificando sinais de overfitting ou underfitting;
- Selecionar o modelo final com base no equilíbrio entre desempenho, robustez e interpretabilidade.

Esta fase é crítica para garantir que o modelo desenvolvido não só apresenta bom desempenho estatístico, mas também gera valor para o negócio, permitindo apoiar decisões estratégicas na gestão de recursos humanos, nomeadamente na retenção de talento.

**Autores: Luís Figueira, Martim Ferreira, Mateus Afonso**

# Índice de Risco - Regressão Logística
Modelo de Regressão Logística com StandardScaler e classificação de risco.

In [21]:
# 1. IMPORTAÇÕES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, RocCurveDisplay,
    accuracy_score, precision_score,
    recall_score, f1_score,
    confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings("ignore")
print("Bibliotecas importadas com sucesso.")

Bibliotecas importadas com sucesso.


In [22]:
# 2. CARREGAMENTO DO DATASET

#url = "https://raw.githubusercontent.com/LuiscnFigueira/Projeto-cdg-grupo10/main/data/processed/dataset_processed.csv"
#df = pd.read_csv(url)
#print(f"Dataset carregado: {df.shape[0]} linhas, {df.shape[1]} colunas")

In [23]:
# 3. PREPARAÇÃO DAS FEATURES

#cols_remover = ["Attrition", "OverTime", "Gender",
#                "BusinessTravel", "Department", "EducationField",
#                "JobRole", "MaritalStatus"]

#cols_remover = [c for c in cols_remover if c in df.columns]
#df_model = df.drop(columns=cols_remover)

#TARGET = "Attrition_bin"
#X = df_model.drop(columns=[TARGET])
#y = df_model[TARGET]
#X = X.select_dtypes(include=[np.number])

#print(f"Features utilizadas: {X.shape[1]}")

In [24]:
# 4. DIVISÃO TREINO / TESTE — Gerar
#import os
#import zipfile
#from IPython.display import FileLink, display

#treino_path = "data/processed/Objetivo2/treino"
#teste_path  = "data/processed/Objetivo2/teste"

# 80% treino, 20% teste
# stratify=y — garante a mesma proporção de Yes/No em treino e teste
#X_train, X_test, y_train, y_test = train_test_split(
#    X, y, test_size=0.2, random_state=42, stratify=y
#)

# Criar as pastas e guardar os splits
#os.makedirs(treino_path, exist_ok=True)
#os.makedirs(teste_path, exist_ok=True)
#X_train.to_csv(f"{treino_path}/X_train.csv", index=False)
#y_train.to_csv(f"{treino_path}/y_train.csv", index=False)
#X_test.to_csv(f"{teste_path}/X_test.csv", index=False)
#y_test.to_csv(f"{teste_path}/y_test.csv", index=False)

# Criar ZIP com a estrutura de pastas completa
#zip_path = "data/processed/Objetivo2/splits.zip"
#with zipfile.ZipFile(zip_path, "w") as zipf:
#    zipf.write(f"{treino_path}/X_train.csv", "treino/X_train.csv")
#    zipf.write(f"{treino_path}/y_train.csv", "treino/y_train.csv")
#    zipf.write(f"{teste_path}/X_test.csv",   "teste/X_test.csv")
#    zipf.write(f"{teste_path}/y_test.csv",   "teste/y_test.csv")

#print("Splits gerados! :")
#display(FileLink(zip_path))

In [25]:
# 4. DIVISÃO TREINO / TESTE — CARREGAR DO GITHUB
base_treino = "https://raw.githubusercontent.com/LuiscnFigueira/Projeto-cdg-grupo10/main/data/processed/Objetivo2/treino"
base_teste  = "https://raw.githubusercontent.com/LuiscnFigueira/Projeto-cdg-grupo10/main/data/processed/Objetivo2/teste"

# Carregar diretamente do GitHub
X_train = pd.read_csv(f"{base_treino}/X_train.csv")
y_train = pd.read_csv(f"{base_treino}/y_train.csv").squeeze()
X_test  = pd.read_csv(f"{base_teste}/X_test.csv")
y_test  = pd.read_csv(f"{base_teste}/y_test.csv").squeeze()

print(f"Treino: {X_train.shape[0]} obs | Teste: {X_test.shape[0]} obs")

URLError: <urlopen error [Errno -3] Temporary failure in name resolution>

In [ ]:
# 5. TREINO — REGRESSÃO LOGÍSTICA

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression())
])

pipeline.fit(X_train, y_train)
print("Modelo treinado.")

In [ ]:
# 6. AVALIAÇÃO NO CONJUNTO DE TREINO E TESTE

def avaliar_modelo(pipeline, X, y, nome_conjunto):
    y_pred       = pipeline.predict(X)
    y_pred_proba = pipeline.predict_proba(X)[:, 1]

    acc       = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall    = recall_score(y, y_pred)
    f1        = f1_score(y, y_pred)
    auc       = roc_auc_score(y, y_pred_proba)

    print(f"\n===== MÉTRICAS — {nome_conjunto} =====")
    print(f"  Accuracy:   {acc:.4f}")
    print(f"  Precision:  {precision:.4f}")
    print(f"  Recall:     {recall:.4f}")
    print(f"  F1-Score:   {f1:.4f}")
    print(f"  AUC-ROC:    {auc:.4f}")

    print(f"\n--- Classification Report ({nome_conjunto}) ---")
    print(classification_report(y, y_pred, target_names=["Ficou", "Saiu"]))

    cm = confusion_matrix(y, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Ficou", "Saiu"],
                yticklabels=["Ficou", "Saiu"], ax=ax)
    ax.set_xlabel("Previsto")
    ax.set_ylabel("Real")
    ax.set_title(f"Matriz de Confusão — {nome_conjunto}")
    plt.tight_layout()
    plt.savefig(f"confusion_matrix_{nome_conjunto.lower()}.png", dpi=150, bbox_inches="tight")
    plt.show()

    return {"conjunto": nome_conjunto, "acc": acc, "precision": precision,
            "recall": recall, "f1": f1, "auc": auc,
            "y": y, "y_proba": y_pred_proba}

resultados_treino = avaliar_modelo(pipeline, X_train, y_train, "Treino")
resultados_teste  = avaliar_modelo(pipeline, X_test,  y_test,  "Teste")

In [ ]:
# 7. COMPARAÇÃO TREINO vs TESTE

print("\n===== COMPARAÇÃO TREINO vs TESTE =====")
print(f"  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}  {'Diferença':>10}")
print(f"  {'-'*42}")
for metrica in ["acc", "precision", "recall", "f1", "auc"]:
    val_treino = resultados_treino[metrica]
    val_teste  = resultados_teste[metrica]
    diff       = val_treino - val_teste
    nome       = metrica.upper() if metrica != "acc" else "Accuracy"
    print(f"  {nome:<12}  {val_treino:>8.4f}  {val_teste:>8.4f}  {diff:>+10.4f}")

print("\n  Nota: Diferença positiva indica possível overfitting.")

In [ ]:
# 8. CURVAS ROC SOBREPOSTAS

fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(
    resultados_treino["y"], resultados_treino["y_proba"],
    name=f"Treino (AUC={resultados_treino['auc']:.3f})", ax=ax, color="steelblue"
)
RocCurveDisplay.from_predictions(
    resultados_teste["y"], resultados_teste["y_proba"],
    name=f"Teste  (AUC={resultados_teste['auc']:.3f})", ax=ax, color="darkorange"
)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_title("Curva ROC — Treino vs Teste")
plt.tight_layout()
plt.savefig("roc_treino_vs_teste.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 9. GERAR PROBABILIDADES DE SAÍDA (dataset completo)

df_risco = df.copy()
df_risco["prob_saida"] = pipeline.predict_proba(X)[:, 1]

print(f"\nProbabilidades geradas para {len(df_risco)} colaboradores.")
print(df_risco["prob_saida"].describe(percentiles=[.25, .50, .75, .90]).round(4))

In [ ]:
# 10. CLASSIFICAÇÃO EM CATEGORIAS DE RISCO

def classificar_risco(prob):
    if prob < 0.30:
        return "Baixo"
    elif prob <= 0.60:
        return "Medio"
    else:
        return "Alto"

df_risco["nivel_risco"] = df_risco["prob_saida"].apply(classificar_risco)

ORDEM = ["Baixo", "Medio", "Alto"]

contagem    = df_risco["nivel_risco"].value_counts()
percentagem = df_risco["nivel_risco"].value_counts(normalize=True) * 100

print("\n===== DISTRIBUICAO DAS CATEGORIAS DE RISCO =====")
print(f"  {'Categoria':<10}  {'Contagem':>8}  {'Percentagem':>12}")
print(f"  {'-'*34}")
for cat in ORDEM:
    n   = contagem.get(cat, 0)
    pct = percentagem.get(cat, 0.0)
    print(f"  {cat:<10}  {n:>8}  {pct:>11.1f}%")

In [ ]:
# 11. ANÁLISE POR CATEGORIA DE RISCO

cols_analise = ["prob_saida", "Attrition_bin"]
for col in ["Age", "MonthlyIncome", "JobLevel", "TotalWorkingYears",
            "OverTime_bin", "SatisfactionIndex", "YearsAtCompany"]:
    if col in df_risco.columns:
        cols_analise.append(col)

print("\n===== PERFIL MEDIO POR CATEGORIA DE RISCO =====")
perfil = df_risco.groupby("nivel_risco")[cols_analise].mean().reindex(ORDEM).round(3)
display(perfil)

In [ ]:
# 12. TOP 20 COLABORADORES COM MAIOR RISCO

cols_top = ["prob_saida", "nivel_risco", "Attrition_bin"]
for col in ["Age", "MonthlyIncome", "JobLevel", "OverTime_bin"]:
    if col in df_risco.columns:
        cols_top.append(col)

top20 = df_risco.nlargest(20, "prob_saida")[cols_top].reset_index(drop=True)

print("\n===== TOP 20 — MAIOR PROBABILIDADE DE SAIDA =====")
display(top20)

In [ ]:
# 13. VISUALIZAÇÕES DO ÍNDICE DE RISCO

# 13.1 Histograma das probabilidades
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df_risco["prob_saida"], bins=40, color="steelblue", edgecolor="white")
ax.axvline(0.30, color="orange", linestyle="--", linewidth=1.5, label="Threshold 30%")
ax.axvline(0.60, color="red",    linestyle="--", linewidth=1.5, label="Threshold 60%")
ax.set_xlabel("Probabilidade de Saida")
ax.set_ylabel("Numero de Colaboradores")
ax.set_title("Distribuicao das Probabilidades de Saida")
ax.legend()
plt.tight_layout()
plt.savefig("distribuicao_probabilidades.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 13.2 Contagem por categoria
cores = {"Baixo": "#2ecc71", "Medio": "#f39c12", "Alto": "#e74c3c"}

fig, ax = plt.subplots(figsize=(7, 5))
vals = [contagem.get(c, 0) for c in ORDEM]
bars = ax.bar(ORDEM, vals, color=[cores[c] for c in ORDEM], edgecolor="white", width=0.5)
for bar, cat in zip(bars, ORDEM):
    n   = contagem.get(cat, 0)
    pct = percentagem.get(cat, 0.0)
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f"{n}\n({pct:.1f}%)",
        ha="center", va="bottom", fontsize=11
    )
ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Numero de Colaboradores")
ax.set_title("Colaboradores por Categoria de Risco")
plt.tight_layout()
plt.savefig("categorias_risco.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 13.3 Boxplot por categoria
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df_risco, x="nivel_risco", y="prob_saida", order=ORDEM, ax=ax)
ax.set_xlabel("Categoria de Risco")
ax.set_ylabel("Probabilidade de Saida")
ax.set_title("Distribuicao das Probabilidades por Categoria")
plt.tight_layout()
plt.savefig("boxplot_risco.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 14. RESUMO FINAL

print("\n" + "=" * 55)
print("RESUMO — Baseline")
print("=" * 55)
print(f"  Modelo:         Regressao Logistica (parâmetros default)")
print(f"  Colaboradores:  {len(df_risco)}")
print(f"\n  {'Métrica':<12}  {'Treino':>8}  {'Teste':>8}")
print(f"  {'-'*32}")
for metrica, nome in [("acc","Accuracy"), ("precision","Precision"),
                       ("recall","Recall"), ("f1","F1-Score"), ("auc","AUC-ROC")]:
    print(f"  {nome:<12}  {resultados_treino[metrica]:>8.4f}  {resultados_teste[metrica]:>8.4f}")
print(f"\n  Baixo risco:    prob < 30%  -> {contagem.get('Baixo', 0)} colaboradores")
print(f"  Risco medio:    30% - 60%   -> {contagem.get('Medio', 0)} colaboradores")
print(f"  Alto risco:     prob > 60%  -> {contagem.get('Alto', 0)} colaboradores")
print("=" * 55)